This jupyter notebook displays the usage of the Interface class that orchestrates information between BMI T-toute and BMI KF. If you want to look into the source code, please refer to Interface class in this directory.

In [ ]:
from Interface import Interface
import copy

yaml_file_path_troute = "../test/tx_res_calibrated/troute.yaml"
yaml_file_path_troute_nudging = "../test/tx_res_calibrated/troute_nudge.yaml"
yaml_file_path_KF = "../test/tx_res_calibrated/KF_Data/KF.yaml"

interface_with_KF = Interface(yaml_file_path_troute=yaml_file_path_troute, yaml_file_path_KF=yaml_file_path_KF)
interface_without_KF = copy.deepcopy(interface_with_KF)
interface_without_KF_nudging = Interface(yaml_file_path_troute=yaml_file_path_troute_nudging, yaml_file_path_KF=yaml_file_path_KF)

In [ ]:
interface_with_KF.warmup_with_KF()

In [ ]:
interface_with_KF.troute_model.get_current_datetime()

In [ ]:
interface_without_KF.warmup_without_KF()

In [ ]:
interface_without_KF_nudging.warmup_without_KF()

In [ ]:
interface_with_KF.troute_model.get_current_datetime()

In [ ]:
interface_without_KF.troute_model.get_current_datetime()

In [ ]:
interface_without_KF_nudging.troute_model.get_current_datetime()

In [ ]:
interface_with_KF.forecast_only()

In [ ]:
interface_without_KF.forecast_only()

In [ ]:
interface_without_KF_nudging.forecast_only()

In [ ]:
#this will give us the dataframe with the results of the forecast simulation
interface_with_KF.get_results_dataframe()

In [ ]:
interface_without_KF.get_results_dataframe()

In [ ]:
interface_without_KF_nudging.get_results_dataframe()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

measurements = pd.read_csv("KF_Data/measurement.csv", index_col=0)
dt = interface_with_KF.KF_model.model_collection.timedelta.seconds
measurements.index = pd.to_datetime(measurements.index)
measurements.columns = measurements.columns.astype(str)

usgs_to_wbid = pd.read_csv("KF_Data/tx_usgs_to_wbid.csv", index_col=0)
usgs_to_wbid["gage_id"] = usgs_to_wbid["gage_id"].astype(str)
usgs_to_wbid["comid"] = usgs_to_wbid["comid"].astype(str)
usgs_to_wbid = pd.Series(
    usgs_to_wbid["comid"].values,
    index=usgs_to_wbid["gage_id"].values,
)

measurements = measurements[usgs_to_wbid.index]
measurements.columns = measurements.columns.map(usgs_to_wbid)

# Subset measurements by the specified time window
measurements = measurements.dropna(axis=1)
measurements = measurements.loc[:, ~(measurements == 0.0).all(axis=0).values]
measurements = measurements.loc[:, ~measurements.columns.duplicated()].copy()
measurements = measurements.resample(f"{dt}s").mean().interpolate().ffill().bfill()


In [ ]:
# excluded_gages = [8155400,8158000]
# included_gages = [8155240,8158035]
# #plot forecasted vs observed for excluded gages
# test_gage = included_gages[0]

In [ ]:
def plot_forecast_vs_observed(obs_df, test_gage, usgs_to_wbid):
    #plot forecast vs observed for with and without KF for a specific gage
    comid = usgs_to_wbid.loc[str(test_gage)]
    simulated_flow_KF = interface_with_KF.get_results_dataframe()[int(comid)]
    simulated_flow_no_KF = interface_without_KF.get_results_dataframe()[int(comid)]
    simulated_flow_no_KF_nudging = interface_without_KF_nudging.get_results_dataframe()[int(comid)]

    observed_flow = obs_df[comid]
    observed_flow.index = pd.to_datetime(observed_flow.index)
    observed_flow = observed_flow[observed_flow.index.isin(simulated_flow_KF.index)]
    print("Observed flow")
    print(observed_flow)
    print("Simulated flow with KF")
    print(simulated_flow_KF)
    print("Simulated flow without KF")
    print(simulated_flow_no_KF)

    plt.figure(figsize=(12, 6))
    plt.plot(simulated_flow_KF.index, simulated_flow_KF.values, label='Simulated Flow (KF)', color='blue')
    plt.plot(simulated_flow_no_KF.index, simulated_flow_no_KF.values, label='Simulated Flow (Only Troute)', color='red')
    plt.plot(simulated_flow_no_KF_nudging.index, simulated_flow_no_KF_nudging.values, label='Simulated Flow (Only Troute with Nudging)', color='purple')
    plt.plot(observed_flow.index, observed_flow.values, label='Observed Flow', color='orange')
    plt.title(f'Forecast vs Observed Flow for Gage {test_gage}')
    plt.xlabel('Date')
    plt.ylabel('Flow (m^3/s)')
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
# plot_forecast_vs_observed(measurements, test_gage, usgs_to_wbid)